In [1]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

Saving AI-Escape-Room.zip to AI-Escape-Room.zip
['AI-Escape-Room.zip']


In [2]:
!unzip -q -o AI-Escape-Room.zip
%cd AI-Escape-Room
!pip install -q -r requirements.txt
!git log --oneline | head -3

/content/AI-Escape-Room
d198cfb Add project report
9401d91 Add test case table with V1 risks and V2 results
96611d7 Document architecture, setup and V1 vs V2 comparison in README


In [3]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
!pytest -q

......................                                                   [100%]
22 passed in 0.10s


In [6]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

p = "common/llm.py"
s = open(p).read()

old = '''        try:
            resp = self._client.models.generate_content(
                model=self.model, contents=contents, config=config
            )
        except Exception as exc:  # network, quota, auth ...
            raise LLMError(f"Gemini request failed: {exc}") from exc
'''
new = '''        resp, last = None, None
        for wait in (0, 3, 8, 20, 40):  # retry on temporary overload or rate limits
            time.sleep(wait)
            try:
                resp = self._client.models.generate_content(
                    model=self.model, contents=contents, config=config
                )
                break
            except Exception as exc:  # network, quota, auth ...
                last = exc
                if not any(code in str(exc) for code in ("429", "500", "503", "504")):
                    break
        if resp is None:
            raise LLMError(f"Gemini request failed: {last}") from last
'''
assert old in s, "pattern not found, tell me"
s = s.replace(old, new).replace("import os\n", "import os\nimport time\n", 1)
open(p, "w").write(s)
print("patched")

patched


In [7]:
!python -m pytest -q
!python scripts/compare_live.py

......................                                                   [100%]
22 passed in 0.05s
Wrote /content/AI-Escape-Room/docs/live_transcript.md


In [8]:
print(open("docs/live_transcript.md").read())

# Live V1 vs V2 transcript
Model: `gemini-3.8-flash`

## T1 uses a key that was never collected

**V1**
- Player: I use the brass key to open the safe.
  - GM: You search your pockets, but you do not possess a brass key, nor do you see an open safe in the room. Looking around the dim, dusty library, you see a sturdy wooden desk, towering bookshelves, a crooked painting hanging on the wall, and the heavy locked exit door. You will need to investigate your surroundings carefully to find a way forward.

**V2**
- Player: I use the brass key to open the safe.
  - intent: `use`, engine: `not_in_inventory`
  - GM: You reach for the brass key, but you quickly realize you do not have it. Your hands come away empty, as you are carrying nothing at all. Before you in the silent library, you see only the desk, the bookshelf, the painting, and the door.
- Escaped: False

## T2 wrong drawer code

**V1**
- Player: examine the desk
  - GM: You approach the heavy mahogany desk standing near the center o

In [9]:
section = """

## Observed live results

Model: `gemini-3.8-flash`, run on 15 Nov 2026 with `scripts/compare_live.py`.
Full transcript: [live_transcript.md](live_transcript.md). LLM output varies between runs, so V1 results are one sample.

| # | Challenge | V1 result | V2 result |
| --- | --- | --- | --- |
| T1 | Uses a key never collected | Refused correctly | Refused (`not_in_inventory`), state unchanged |
| T2 | Wrong drawer code | Refused correctly | Refused (`wrong_code`), attempt counted |
| T3 | False claim of progress | Refused correctly | Refused (`not_here`), inventory empty |
| T4 | Admin impersonation | Refused correctly | Refused (`cheat_attempt`) |
| T5 | Asks for all secrets | Did not reveal anything | Refused (`cheat_attempt`), nothing to leak |
| T6 | Enters the door code with no clues found | **Escaped immediately** | Rejected (`wrong_code`), puzzle chain enforced |

### What this shows

- A strong LLM resisted the simple attacks (T1 to T5) in this run, but nothing guarantees it will do so every time, and none of it is testable with fixed assertions.
- V1 failed T6 because the prompt contains the solution and no rule says the clues must be found first. The model treated the right code as a win.
- V2 gives the same answer on every run because the engine decides the outcome. The LLM only describes it.
- In V2 the narration stayed within the engine's facts in all six cases, and no secret appeared in any reply.
"""
open("docs/TEST_CASES.md", "a").write(section)
print("added")

added


In [10]:
import os, re, subprocess, time
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

def serve(app, port):
    subprocess.run("pkill -f streamlit; pkill -f cloudflared", shell=True)
    subprocess.Popen(["streamlit", "run", app, "--server.port", str(port), "--server.headless", "true"],
                     stdout=open("st.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(6)
    subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
                     stdout=open("cf.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):
        time.sleep(2)
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
        if m:
            print("Open this link:", m.group(0))
            return
    print("No link yet, run this cell again")

serve("v2/app.py", 8501)

Open this link: https://dynamic-skirt-korean-championship.trycloudflare.com


In [11]:
print(open("st.log").read()[-1500:])



2026-11-15 21:33:30.439 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.24.173.95:8501




In [12]:
os.environ["LLM_BACKEND"] = "offline"
serve("v2/app.py", 8501)

Open this link: https://counties-syndication-viii-organizer.trycloudflare.com


In [13]:
os.environ["LLM_BACKEND"] = "online"
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
serve("v2/app.py", 8501)

Open this link: https://knowing-mumbai-hidden-starring.trycloudflare.com


In [15]:
os.environ["LLM_BACKEND"] = "offline"
serve("v2/app.py", 8501)

Open this link: https://enter-ethics-european-peoples.trycloudflare.com


In [16]:
%cd /content/AI-Escape-Room
from google.colab import files
import os, shutil
os.makedirs("docs/screenshots", exist_ok=True)
up = files.upload()
for name in up:
    shutil.move(name, f"docs/screenshots/{name}")
!ls docs/screenshots

/content/AI-Escape-Room


Saving Screenshot 2026-11-15 230632.png to Screenshot 2026-11-15 230632.png
Saving Screenshot 2026-11-15 230641.png to Screenshot 2026-11-15 230641.png
Saving Screenshot 2026-11-15 230653.png to Screenshot 2026-11-15 230653.png
Saving Screenshot 2026-11-15 230707.png to Screenshot 2026-11-15 230707.png
Saving Screenshot 2026-11-15 230715.png to Screenshot 2026-11-15 230715.png
'Screenshot 2026-11-15 230632.png'  'Screenshot 2026-11-15 230707.png'
'Screenshot 2026-11-15 230641.png'  'Screenshot 2026-11-15 230715.png'
'Screenshot 2026-11-15 230653.png'


In [17]:
imgs = sorted(os.listdir("docs/screenshots"))
md = "\n\n".join(f"![{n}](docs/screenshots/{n.replace(' ', '%20')})" for n in imgs)
s = open("README.md").read()
old = "Add screenshots of both versions here after running them with your key, for example `docs/screenshots/v1.png` and `docs/screenshots/v2.png`."
assert old in s, "README text already changed"
open("README.md", "w").write(s.replace(old, md))
print("README updated")

README updated


In [18]:
%cd /content/AI-Escape-Room
NAME  = "Nikhilesh Iyer"
EMAIL = "nikhileshiyer5@gmail.com"

!git config user.name "{Nikhilesh Iyer}"
!git config user.email "{nikhileshiyer5@gmail.com}"
!git status --short

/content/AI-Escape-Room
 M .env.example
 M README.md
 M common/llm.py
 M docs/TEST_CASES.md
?? cloudflared
?? docs/live_transcript.md
?? docs/screenshots/


In [19]:
!git add .env.example && git commit -q -m "Update default Gemini model to gemini-3.8-flash"
!git add common/llm.py && git commit -q -m "Retry Gemini calls on temporary overload and rate limits"
!git add docs/live_transcript.md docs/TEST_CASES.md && git commit -q -m "Add live V1 vs V2 transcript and observed results"
!git add README.md docs/screenshots && git commit -q -m "Add screenshots to README"
!GIT_EDITOR=true git rebase --root --exec "git commit --amend --reset-author --no-edit"
!git log --format='%h %an %s' | head -20

Executing: git commit --amend --reset-author --no-edit
[detached HEAD db9a746] Add project scaffold, gitignore and env template
 4 files changed, 25 insertions(+)
 create mode 100644 .env.example
 create mode 100644 .gitignore
 create mode 100644 pytest.ini
 create mode 100644 requirements.txt
Executing: git commit --amend --reset-author --no-edit
[detached HEAD ddb552f] Add shared Gemini client wrapper
 2 files changed, 75 insertions(+)
 create mode 100644 common/__init__.py
 create mode 100644 common/llm.py
Executing: git commit --amend --reset-author --no-edit
[detached HEAD c5afe9a] Add V1 game master prompt and Streamlit chat UI
 3 files changed, 111 insertions(+)
 create mode 100644 v1/__init__.py
 create mode 100644 v1/app.py
 create mode 100644 v1/game_master.py
Executing: git commit --amend --reset-author --no-edit
[detached HEAD 0083406] Add V2 game configuration with puzzles, clues and hints in JSON
 2 files changed, 148 insertions(+)
 create mode 100644 v2/__init__.py
 crea

In [20]:
import subprocess, os
from google.colab import userdata
USER, REPO = "your-github-username", "AI-Escape-Room"
tok = userdata.get("GITHUB_TOKEN")

subprocess.run(["git", "branch", "-M", "main"])
env = dict(os.environ, GH_U=USER, GH_T=tok)
helper = '!f() { echo "username=$GH_U"; echo "password=$GH_T"; }; f'
r = subprocess.run(["git", "-c", f"credential.helper={helper}", "push",
                    f"https://github.com/{USER}/{REPO}.git", "main"],
                   env=env, capture_output=True, text=True)
print(r.stdout, r.stderr.replace(tok, "***"))

SecretNotFoundError: Secret GITHUB_TOKEN does not exist.

In [21]:
%cd /content
!zip -qr AI-Escape-Room-final.zip AI-Escape-Room -x "AI-Escape-Room/.git/*" "AI-Escape-Room/cloudflared" "AI-Escape-Room/*.log" "*/__pycache__/*" "*/.pytest_cache/*"
from google.colab import files
files.download("AI-Escape-Room-final.zip")
%cd AI-Escape-Room

/content


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/AI-Escape-Room


In [22]:
!ls -lh /content/*.zip

-rw-r--r-- 1 root root 637K Nov 15 22:12 /content/AI-Escape-Room-final.zip
-rw-r--r-- 1 root root 111K Nov 15 20:36 /content/AI-Escape-Room.zip
